# Analyse der Traces von der DDOS-Attacke auf den mobi3-Server

## Imports

In [3]:
import subprocess
import io
from pathlib import Path
from typing import cast
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from datetime import datetime 

## Global Variables

In [4]:
DEST_IP = '141.22.28.227'

## 1. Files to dataframe

In [ ]:
def pcap_to_df(file: Path) -> pl.DataFrame:
    output_name = file.stem + ".csv"
    tshark_cmd = [
        "tshark",
        "-r", str(file),
        "-E", "header=y",
        "-E", "separator=,",
        "-E", "quote=d",
    ]
    
    with open(output_name, "w") as f:
        subprocess.run(tshark_cmd, stdout=f)

    df = pl.read_csv(output_name, )
    return df

In [15]:
file = Path("data/2025-08-13_15-38CEST_ddos-event_full-packets_100MiB.pcap.zst")
df_raw = pcap_to_df(file=file)
print(df_raw.shape)
df_raw.show(30)

AttributeError: 'int' object has no attribute 'head'

## 2. Aufbereitung

### Relevant ist nur eingehender Traffic

In [ ]:
df_in = df_raw.filter(
    pl.col("ip.dst") == DEST_IP
)

print(df_in.shape)

### IANA Protocols Liste einlesen

In [ ]:
df_pro_nb = (
    pl.read_csv("data/protocol-numbers-1.csv", ignore_errors=True)
      .select("Decimal", "Keyword")
).rename({"Keyword": "ip_proto_name", "Decimal": "ip_proto_num"})
df_pro_nb.head(10)

In [ ]:
df = (
    df_in
    .rename({c: c.replace(".", "_") for c in df_in.columns})  # Punkte → Unterstriche
    .with_columns([
        # Timestamp als datetime
        (pl.col("frame_time_epoch").cast(pl.Float64) * 1_000_000)
            .cast(pl.Int64)
            
            .cast(pl.Datetime("us"))
            .alias("timestamp"),

        # Einheitliche Ports (TCP oder UDP)
        pl.coalesce(["tcp_srcport", "udp_srcport"]).alias("src_port"),
        pl.coalesce(["tcp_dstport", "udp_dstport"]).alias("dst_port"),
    ])
    # translate number to name with iana protocol list
    .join(df_pro_nb, left_on="ip_proto", right_on="ip_proto_num", how="left")

    .drop(["frame_time_epoch", "ip_proto", "tcp_srcport", "tcp_dstport", "udp_srcport", "udp_dstport"])
)

print(df.shape)
df.show(10)

df_special = df.filter(
    pl.col("ip_proto_name") == "TCP"
    # ~pl.col("ip_proto_name").is_in(["tcp", "udp"])
)
print(df.shape)
df_special.show(10)


## 3. Überblick

In [ ]:
total_packets: int  = df_raw.height
total_bytes: int    = int(df_raw["frame_len"].sum())

t_start = cast(datetime, df_raw["timestamp"].min())
t_end   = cast(datetime, df_raw["timestamp"].max())
duration_s = (t_end - t_start).total_seconds()

print(f"Pakete gesamt : {total_packets:>12,}")
print(f"Bytes gesamt  : {total_bytes:>12,}  ({total_bytes/1e6:.1f} MB)")
print(f"Zeitraum      : {t_start}  →  {t_end}")
print(f"Dauer         : {duration_s:.1f} s")
print(f"Ø Paketrate   : {total_packets/duration_s:>10,.1f} pkt/s")
print(f"Ø Bitrate     : {total_bytes*8/duration_s/1e6:>10,.1f} Mbit/s")

## 4. Protokollverteilung

In [ ]:
proto_dist = (
    df.with_columns(pl.col("l4_proto").fill_null("unknown"))
      .group_by("l4_proto")
      .agg([
          pl.len().alias("packets"),
          pl.col("frame_len").sum().alias("bytes"),
      ])
      .sort("packets", descending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ["packets", "bytes"], ["Pakete", "Bytes"]):
    ax.bar(proto_dist["l4_proto"], proto_dist[col])
    ax.set_title(f"Protokoll – {label}")
    ax.set_xlabel("Protokoll")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

proto_dist

## 5. Zeitreihe – Paketrate

In [ ]:
BUCKET = "1s"  # Aggregations-Intervall: "100ms", "1s", "10s" ...

ts_rate = (
    df.sort("timestamp")
      .group_by_dynamic("timestamp", every=BUCKET)
      .agg([
          pl.len().alias("pkt_per_s"),
          (pl.col("frame_len").sum() * 8 / 1e6).alias("mbit_per_s"),
      ])
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(ts_rate["timestamp"].to_list(), ts_rate["pkt_per_s"].to_list(), linewidth=0.8)
ax1.set_ylabel("Pakete/s")
ax1.set_title(f"Paketrate (Bucket: {BUCKET})")

ax2.plot(ts_rate["timestamp"].to_list(), ts_rate["mbit_per_s"].to_list(), color="orange", linewidth=0.8)
ax2.set_ylabel("Mbit/s")
ax2.set_xlabel("Zeit")

plt.tight_layout()
plt.show()

## 6. Top Quell-IPs (potenzielle Angreifer)

In [ ]:
TOP_N = 20

top_src = (
    df.filter(pl.col("ip_src").is_not_null())
      .group_by("ip_src")
      .agg([
          pl.len().alias("packets"),
          pl.col("frame_len").sum().alias("bytes"),
          pl.col("ip_dst").n_unique().alias("unique_dst"),
      ])
      .sort("packets", descending=True)
      .head(TOP_N)
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_src["ip_src"].to_list()[::-1], top_src["packets"].to_list()[::-1])
ax.set_xlabel("Pakete")
ax.set_title(f"Top {TOP_N} Quell-IPs")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

top_src

## 7. Top Ziel-Ports

In [ ]:
top_dstport = (
    df.filter(pl.col("dst_port").is_not_null())
      .group_by(["dst_port", "l4_proto"])
      .agg(pl.len().alias("packets"))
      .sort("packets", descending=True)
      .head(15)
      .with_columns(
          (pl.col("dst_port").cast(pl.Utf8) + "/" + pl.col("l4_proto")).alias("port_proto")
      )
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top_dstport["port_proto"].to_list(), top_dstport["packets"].to_list())
ax.set_xlabel("Port/Protokoll")
ax.set_ylabel("Pakete")
ax.set_title("Top Ziel-Ports")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

top_dstport.drop("port_proto")

## 8. Paketgrößenverteilung

In [ ]:
sizes = df["frame_len"].drop_nulls().to_list()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(sizes, bins=100, edgecolor="none")
ax.set_xlabel("Paketgröße (Bytes)")
ax.set_ylabel("Häufigkeit")
ax.set_title("Paketgrößenverteilung")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

df.select(pl.col("frame_len").describe())

## 9. Export

In [ ]:
out = Path("output")
out.mkdir(exist_ok=True)

df.write_parquet(out / "packets.parquet")
top_src.write_csv(out / "top_src_ips.csv")

print("Exportiert nach:", out.resolve())